In [ ]:
# Capstone Data Pipeline

# Setup Code

import ee          # Import Google Earth Engine library
import time        # Import time library (used later for delays)

# Define datasets to be used (radar + optical satellite data)
datasets = {
    'S1': 'COPERNICUS/S1_GRD',                 # Sentinel-1 (radar imagery)
    'S2': 'COPERNICUS/S2_SR_HARMONIZED'        # Sentinel-2 (optical imagery)
}

# Define folders where output data will be stored
folders = {
    'S1': 'GEE Data Set',
    'S2': 'GEE Data Set'
}

# Define target dates for analysis (time periods of interest)
dates = ['2024-06-01', '2024-07-01', '2024-08-01']

# Authenticate user with Google Earth Engine
ee.Authenticate()

# Initialize Earth Engine session (required before using data)
ee.Initialize()

# Load global country boundary dataset
bangladesh = ee.FeatureCollection("FAO/GAUL/2015/level0") \
    .filter(ee.Filter.eq('ADM0_NAME', 'Bangladesh'))  
    # Filter dataset to only include Bangladesh

# Create region of interest (ROI)
region = bangladesh.geometry().centroid().buffer(50000)
# Get center point of Bangladesh, create a 50km buffer around the center (defines analysis area)

# Extract coordinates of the region for verification/debugging
coords = region.coordinates().getInfo()

# Print coordinates to confirm region selection
print(coords)

# Print confirmation that setup is complete
print('Initialization Completed.')


Successfully saved authorization token.
[[[90.28837386710796, 24.276995288806475], [90.15052767046068, 24.259026125077085], [90.02376242418443, 24.20656350542334], [89.91824245169275, 24.12382235971146], [89.84238309983382, 24.01743970357822], [89.80217634229456, 23.895930188447696], [89.80072912317, 23.768993743978754], [89.83804585340424, 23.64673501099813], [89.91106264878921, 23.53885918799854], [90.01391978743543, 23.453908021537572], [90.13844219176144, 23.398593945712854], [90.27478584374819, 23.37728100939188], [90.41220033614272, 23.39164933445357], [90.5398533021645, 23.44056633428994], [90.64766039956298, 23.52017350244235], [90.72706434978717, 23.624182840559563], [90.77170820323681, 23.74436246919804], [90.77795193423826, 23.871177222667512], [90.74518846296306, 23.994537751810554], [90.6759262118685, 24.10460171471373], [90.57562103107436, 24.1925641025644], [90.45226065674973, 24.25137181929576], [90.31572836698558, 24.27630136409888], [90.28837386710796, 24.27699528880

In [ ]:
# S1 Data Collection

def cleanSAR(image):
    # Convert raw SAR values to decibels (log scale)
    image = image.log10().multiply(10)
    
    # Clamp values to remove extreme noise (-30 to 5 dB range)
    image = image.clamp(-30, 5)
    
    # Mask out very low values (likely noise)
    image = image.updateMask(image.gt(-30))
    
    # Apply smoothing filter (reduce speckle noise)
    image = image.focal_mean(radius=60, units='meters')
    
    # Normalize values to range 0–1
    image = image.unitScale(-30, 5)
    
    # Return cleaned image
    return image


# Loop through each target date
for date in dates:
    
    # Load Sentinel-1 image collection
    raw_collection = (
        ee.ImageCollection(datasets['S1'])
        .filterBounds(region)                 # Only images in region
        .filterDate(
            ee.Date(date).advance(-20, 'day'),
            ee.Date(date).advance(20, 'day')
        )                                     # Only images near date
        .filter(ee.Filter.eq('instrumentMode', 'IW'))  # Keep correct mode
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))  # Keep VV
        .select(['VV'])                        # Select VV band
    )

    # Count how many images were found
    count = raw_collection.size().getInfo()
    print(f"{date} image count:", count)

    # Skip if no images available
    if count == 0:
        print(f"{date} skipped — no data")
        continue

    # Apply cleaning function to all images
    collection = raw_collection.map(cleanSAR)

    # Combine images into one (median composite)
    image = collection.median()

    # Export image to Google Drive
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=f"S1_{date}",
        folder=folders['S1'],   
        fileNamePrefix=f"S1_{date}",
        region=region,
        scale=10,               # 10m pixel resolution
        maxPixels=1e13
    )

    # Start export task
    task.start()

    # Print task status
    status = task.status()
    print(f"{date} finished with status: {status['state']}")

print("Finished!")

2024-06-01 image count: 16
2024-06-01 finished with status: READY
2024-07-01 image count: 11
2024-07-01 finished with status: READY
2024-08-01 image count: 10
2024-08-01 finished with status: READY
Finished!


In [ ]:
# S2 Data Collection

def cleanS2(image):
    # Extract cloud mask band
    qa = image.select('QA60')
    
    # Define cloud bit (bit 10)
    cloudBitMask = 1 << 10

    # Create mask where clouds are removed
    mask = qa.bitwiseAnd(cloudBitMask).eq(0)
    
    # Apply mask and scale reflectance values
    image = image.updateMask(mask).divide(10000)
    
    # Calculate NDVI (vegetation index)
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    
    # Calculate NDWI (water index)
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    
    # Add indices as new bands
    image = image.addBands(ndwi)
    image = image.addBands(ndvi)

    # Return cleaned image
    return image


# Loop through each target date
for date in dates:

    # Load Sentinel-2 image collection
    collection = (
        ee.ImageCollection(datasets['S2'])
        .filterBounds(region)                 # Filter to region
        .filterDate(
            ee.Date(date).advance(-20, 'day'),
            ee.Date(date).advance(20, 'day')
        )                                     # Filter by date range
        .select(['B3', 'B4', 'B8', 'QA60'])    # Select required bands
        .map(cleanS2)                          # Apply cleaning function
        .select(['B4', 'NDVI', 'NDWI'])        # Keep useful bands
    )
    
    # Check if images exist
    size = collection.size().getInfo()
    
    # Skip if no data
    if size == 0:
        continue

    # Combine images into one (median composite)
    image = collection.median()

    # Export image to Google Drive
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=f"S2_{date}",
        folder=folders['S2'],
        fileNamePrefix=f"S2_{date}",
        region=region, 
        scale=10,               # 10m resolution
        maxPixels=1e13
    )

    # Start export
    task.start()

    # Print status
    status = task.status()
    print(f"{date} finished with status: {status['state']}")

print("Finished!")

2024-06-01 finished with status: READY
2024-07-01 finished with status: READY
2024-08-01 finished with status: READY
Finished!
